# ARC NeuroGolf static ONNX solver 09- localish_recolor

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
required={'onnx':'onnx','onnxruntime':'onnxruntime','onnxsim':'onnxsim','torch':'torch','numpy':'numpy'}
missing=[pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])


import json, os, random, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
import onnxruntime as ort
from onnxsim import simplify

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 77.6 MB/s eta 0:00:00


In [4]:
TASK_ID='task287'; CH=10; H=W=30

In [5]:
class D4Majority16Static(nn.Module):
    def forward(self,x):
        core=x[:,:,:16,:16]
        tr=core.transpose(2,3)
        counts=core+torch.flip(core,[3])+torch.flip(core,[2])+torch.flip(core,[2,3])+tr+torch.flip(tr,[3])+torch.flip(tr,[2])+torch.flip(tr,[2,3])
        pred=torch.argmax(counts,dim=1,keepdim=True)
        core_out=torch.cat([(pred==k).float() for k in range(CH)],1)
        zright = x[:, :, :16, 16:30] * 0.0
        top = torch.cat([core_out, zright], 3)
        zbottom = x[:, :, 16:30, :] * 0.0
        return torch.cat([top, zbottom], 2)
def load_task():
    for p in [Path.cwd()/f'{TASK_ID}.json', Path.cwd().parent/f'{TASK_ID}.json', 
              Path('/mnt/data')/f'{TASK_ID}.json',Path(COMPETITION)/f'{TASK_ID}.json']:
        if p.exists(): return json.load(open(p)), p
    raise FileNotFoundError(TASK_ID)
def onehot(grid):
    a=np.array(grid); h,w=a.shape
    x=np.zeros((1,CH,H,W),dtype=np.float32)
    for k in range(CH): x[0,k,:h,:w]=(a==k)
    return x,h,w

In [6]:
OUT=Path.cwd()/'generated_models'; 
OUT.mkdir(exist_ok=True)
task, task_path=load_task(); 

model_path=OUT/f'{TASK_ID}.onnx'
torch.onnx.export(D4Majority16Static().eval(), torch.zeros(1,CH,H,W), str(model_path), input_names=['input'], output_names=['output'], opset_version=13, dynamo=False)
model=onnx.load(str(model_path)); ops={}

/tmp/ipykernel_16/753654272.py:6: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(D4Majority16Static().eval(), torch.zeros(1,CH,H,W), str(model_path), input_names=['input'], output_names=['output'], opset_version=13, dynamo=False)


In [7]:
for node in model.graph.node: ops[node.op_type]=ops.get(node.op_type,0)+1
forbidden=[op for op in ['Loop','Scan','NonZero','Unique','Script','Function'] if ops.get(op,0)]
risk=[op for op in ['ScatterND','Shape','Range','Expand','Gather','ConstantOfShape','Resize','Tile','If'] if ops.get(op,0)]
print('size', model_path.stat().st_size, 'forbidden', forbidden, 'risk', risk, 'ops', ops)
assert model_path.stat().st_size < 1_400_000 and not forbidden and not risk

size 8215 forbidden [] risk [] ops {'Constant': 52, 'Slice': 10, 'Transpose': 1, 'Add': 7, 'ArgMax': 1, 'Equal': 10, 'Cast': 10, 'Concat': 3, 'Mul': 2}


In [8]:
sess=ort.InferenceSession(str(model_path), providers=['CPUExecutionProvider'])
summary={}
for sec in ['train','test','arc-gen']:
    exact=0
    for ex in task[sec]:
        x,h,w=onehot(ex['input'])
        pred=sess.run(None, {'input':x})[0].argmax(1)[0,:h,:w]
        exact += int(np.array_equal(pred,np.array(ex['output'])))
    summary[sec]={'exact':exact,'total':len(task[sec])}
print(summary)
assert all(v['exact']==v['total'] for v in summary.values())


{'train': {'exact': 4, 'total': 4}, 'test': {'exact': 1, 'total': 1}, 'arc-gen': {'exact': 262, 'total': 262}}


In [9]:
with zipfile.ZipFile(Path.cwd()/'submission.zip','w',zipfile.ZIP_DEFLATED) as zf: zf.write(model_path,'task287.onnx')
print('submission.zip written')

submission.zip written
